# Shoe Order Extraction with Replicate API

**Author:** Vijay Sharma  
**Date:** April 7, 2026  
**Course:** DSC 670 - 4.2 Week 4 Exercise

## Overview
This notebook demonstrates zero-shot and one-shot prompting to extract shoe order information from emails using the Replicate API. We'll use Llama 2 or Mistral models to extract customer details as JSON.

## Setup and Installation

First, install the required libraries and set up your Replicate API key.

In [12]:
# Install required packages
%pip install replicate python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [13]:
# Import required libraries
import os
import json
import replicate
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# OPTION 1: Set API key directly here
replicate_api_key = "r8_0w0IxUC5yIRoWw7WdfTpQVEZEoCQxsv144uBQ"

# OPTION 2: Load from .env file (comment out OPTION 1 if using this)
# replicate_api_key = os.getenv("REPLICATE_API_TOKEN")

# Set the API key
os.environ["REPLICATE_API_TOKEN"] = replicate_api_key

print("Libraries imported successfully!")
print(f"Replicate API Key loaded: {'Yes' if replicate_api_key and replicate_api_key != 'r8_0w0IxUC5yIRoWw7WdfTpQVEZEoCQxsv144uBQ' else 'No (placeholder)'}")

Libraries imported successfully!
Replicate API Key loaded: Yes


## Define Prompts

We'll create two prompts:
1. **Zero-shot prompt**: Extract information from the first email without examples
2. **One-shot prompt**: Use the first email as an example to extract from the second email

In [14]:
# Zero-shot prompt for the first email
zero_shot_prompt = """Extract all information from the following email as JSON only. Use these fixed shoe prices:
- Nike Air Jordan I: 100
- Converse All-Star: 30
- New Balance 990: 110
- Nike Zoom Fly 5 men's red: 400

Return:
- shoe_agent_name
- request_type
- items: list of objects with shoe_brand, shoe_model, shoe_quantity, shoe_price, shoe_subtotal
- customer_name
- customer_street
- customer_city
- customer_state
- customer_zip
- customer_phone
- grand_total

Do not add any explanation, markdown, or text outside the JSON.

Email:
Hey Jim,

I'd like to order some new shoes. Please ship the following:

1. Nike Air Jordan I - 2 pair
2. Converse All-Star - 10 pair
3. New Balance 990 - 1 pair
4. Nike Zoom Fly 5 men's red - 2 pair

Please provide sub-totals and grand total cost.

Thanks.

Lance Gentry
123 Main St.
Chelsea, MI 48109
248-229-2229
"""

# One-shot prompt with example for the second email
one_shot_prompt = """Extract the requested information from the second email as JSON only. Use these fixed shoe prices:
- Nike Air Jordan I: 100
- Converse All-Star: 30
- New Balance 990: 110
- Nike Zoom Fly 5 men's red: 400

Return:
- shoe_agent_name
- request_type
- items: list of objects with shoe_brand, shoe_model, shoe_quantity, shoe_price, shoe_subtotal
- customer_name
- customer_street
- customer_city
- customer_state
- customer_zip
- customer_phone
- grand_total

Example:

Email:
Hey Jim,

I'd like to order some new shoes. Please ship the following:

1. Nike Air Jordan I - 2 pair
2. Converse All-Star - 10 pair
3. New Balance 990 - 1 pair
4. Nike Zoom Fly 5 men's red - 2 pair

Please provide sub-totals and grand total cost.

Thanks.

Lance Gentry
123 Main St.
Chelsea, MI 48109
248-229-2229

JSON:
{
  "shoe_agent_name": "Jim",
  "request_type": "Order",
  "items": [
    {
      "shoe_brand": "Nike",
      "shoe_model": "Air Jordan I",
      "shoe_quantity": 2,
      "shoe_price": 100,
      "shoe_subtotal": 200
    },
    {
      "shoe_brand": "Converse",
      "shoe_model": "All-Star",
      "shoe_quantity": 10,
      "shoe_price": 30,
      "shoe_subtotal": 300
    },
    {
      "shoe_brand": "New Balance",
      "shoe_model": "990",
      "shoe_quantity": 1,
      "shoe_price": 110,
      "shoe_subtotal": 110
    },
    {
      "shoe_brand": "Nike",
      "shoe_model": "Zoom Fly 5 men's red",
      "shoe_quantity": 2,
      "shoe_price": 400,
      "shoe_subtotal": 800
    }
  ],
  "grand_total": 1410,
  "customer_name": "Lance Gentry",
  "customer_street": "123 Main St.",
  "customer_city": "Chelsea",
  "customer_state": "MI",
  "customer_zip": "48109",
  "customer_phone": "248-229-2229"
}

Now extract from this email:

Hey Michelle,

I'd like to order some new shoes. Please ship the following:

1. Nike Air Jordan I - 1 pair
2. Converse All-Star - 20 pair
3. New Balance 990 - 2 pair
4. Nike Zoom Fly 5 men's red - 5 pair

Please provide sub-totals and grand total cost.

Thanks.

Artis Gilmore
723 Lexington Blvd.
New York, NY 10001
(503) 484-1029
"""

print("Prompts defined successfully!")

Prompts defined successfully!


## Helper Function

This function will call the Replicate API with the given prompt.

In [17]:
def run_replicate_completion(prompt: str, model: str = "mistral") -> str:
    """
    Run Replicate model inference with the given prompt.
    
    Args:
        prompt (str): The prompt to send to the model
        model (str): The model to use ('mistral', 'llama-2-70b', 'llama-2-13b')
    
    Returns:
        str: The response content
    """
    # Available models on Replicate
    models = {
        "mistral": "mistralai/mistral-7b-instruct-v0.1",
        "llama-2-70b": "meta/llama-2-70b-chat",
        "llama-2-13b": "meta/llama-2-13b-chat",
    }
    
    model_id = models.get(model, models["llama-2-13b"])
    
    print(f"Using model: {model_id}")
    
    output = replicate.run(
        model_id,
        input={
            "prompt": prompt,
            "temperature": 0.1,
            "max_new_tokens": 1000,
            "top_p": 0.9,
        }
    )
    
    # Handle different output formats from Replicate
    if isinstance(output, list):
        return "".join(output)
    return str(output)

print("Helper function defined successfully!")

Helper function defined successfully!


## Run Zero-Shot Prompt

Extract information from the first email using zero-shot prompting (no examples).

In [19]:
print("=== Zero-Shot Prompt Result ===")
try:
    zero_shot_result = run_replicate_completion(zero_shot_prompt, model="llama-2-13b")
    print(zero_shot_result)
    
    # Try to parse and pretty print the JSON
    try:
        parsed_json = json.loads(zero_shot_result)
        print("\n--- Pretty Printed JSON ---")
        print(json.dumps(parsed_json, indent=2))
    except json.JSONDecodeError:
        print("\n[Note: Response couldn't be parsed as JSON - model may have included extra text]")
    
except Exception as e:
    print(f"Zero-shot extraction failed: {e}")

=== Zero-Shot Prompt Result ===
Using model: meta/llama-2-13b-chat
 Sure! Here is the information from the email as JSON:

{
"shoe_agent_name": null,
"request_type": "order",
"items": [
{
"shoe_brand": "Nike",
"shoe_model": "Air Jordan I",
"shoe_quantity": 2,
"shoe_price": 100,
"shoe_subtotal": 200
},
{
"shoe_brand": "Converse",
"shoe_model": "All-Star",
"shoe_quantity": 10,
"shoe_price": 30,
"shoe_subtotal": 300
},
{
"shoe_brand": "New Balance",
"shoe_model": "990",
"shoe_quantity": 1,
"shoe_price": 110,
"shoe_subtotal": 110
},
{
"shoe_brand": "Nike",
"shoe_model": "Zoom Fly 5 men's red",
"shoe_quantity": 2,
"shoe_price": 400,
"shoe_subtotal": 800
}
],
"customer_name": "Lance Gentry",
"customer_street": "123 Main St.",
"customer_city": "Chelsea",
"customer_state": "MI",
"customer_zip": "48109",
"customer_phone": "248-229-2229",
"grand_total": 1110
}

[Note: Response couldn't be parsed as JSON - model may have included extra text]


## Run One-Shot Prompt

Extract information from the second email using one-shot prompting (with example from first email).

In [20]:
print("=== One-Shot Prompt Result ===")
try:
    one_shot_result = run_replicate_completion(one_shot_prompt, model="llama-2-13b")
    print(one_shot_result)
    
    # Try to parse and pretty print the JSON
    try:
        parsed_json = json.loads(one_shot_result)
        print("\n--- Pretty Printed JSON ---")
        print(json.dumps(parsed_json, indent=2))
    except json.JSONDecodeError:
        print("\n[Note: Response couldn't be parsed as JSON - model may have included extra text]")
    
except Exception as e:
    print(f"One-shot extraction failed: {e}")

=== One-Shot Prompt Result ===
Using model: meta/llama-2-13b-chat
 Sure, I'd be happy to help! Here is the information extracted from the email as JSON:

{
"shoe_agent_name": "Michelle",
"request_type": "Order",
"items": [
{
"shoe_brand": "Nike",
"shoe_model": "Air Jordan I",
"shoe_quantity": 1,
"shoe_price": 100,
"shoe_subtotal": 100
},
{
"shoe_brand": "Converse",
"shoe_model": "All-Star",
"shoe_quantity": 20,
"shoe_price": 30,
"shoe_subtotal": 600
},
{
"shoe_brand": "New Balance",
"shoe_model": "990",
"shoe_quantity": 2,
"shoe_price": 110,
"shoe_subtotal": 220
},
{
"shoe_brand": "Nike",
"shoe_model": "Zoom Fly 5 men's red",
"shoe_quantity": 5,
"shoe_price": 400,
"shoe_subtotal": 2000
}
],
"grand_total": 2820,
"customer_name": "Artis Gilmore",
"customer_street": "723 Lexington Blvd.",
"customer_city": "New York",
"customer_state": "NY",
"customer_zip": "10001",
"customer_phone": "(503) 484-1029"
}

I hope this helps! Let me know if you have any other questions.

[Note: Response couldn

## Analysis and Comparison

### Zero-Shot vs One-Shot Prompting

**Zero-Shot Prompting:**
- *Zero shot took little bit time to run and provide the result*

**One-Shot Prompting:**
- *where as one shot prompted the result very fast and accurate.*

**Comparison with OpenAI:**
- *I was unable to run through OPEN AI and also purchased the tokens for replicate due to last project since using that as do not want to spend money in all the Open AI accounts*

### Model Comparison

You can experiment with different models:
- **Mistral 7B**: Faster, good for general tasks
- **Llama 2 13B**: Medium-sized, good balance
- **Llama 2 70B**: Largest, best quality but slower

Change the model parameter in cells 10 and 12 to test different models.

## Next Steps

1. **Get Replicate API Key**: https://replicate.com/account/api-tokens
2. **Update the API key** in cell 4 with your actual key
3. **Run the cells** to extract the shoe order information
4. **Compare results** between Replicate models and OpenAI
5. **Test different models** to see which works best for your use case

Good luck!